# End-to-End Demo
This notebook walks through each layer of the pipeline in sequence:
1. Raw JSON ingestion output
2. Processed Parquet data
3. Analytics outputs

In [1]:
from pyspark.sql import SparkSession
import json
import os
import glob

BASE_PATH = "/Users/snehamungre/projects/crypto_market_analysis"

spark = SparkSession.builder \
    .appName("CryptoDemo") \
    .config("spark.sql.warehouse.dir", f"{BASE_PATH}/spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session started successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/29 12:57:15 WARN Utils: Your hostname, Sneehes-Mac.local, resolves to a loopback address: 127.0.0.1; using 192.0.0.2 instead (on interface en0)
26/05/29 12:57:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 12:57:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/29 12:57:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark session started successfully


---
## Stage 1 — Raw Ingestion Output

In [13]:
raw_files = sorted(glob.glob(f"{BASE_PATH}/data/raw/*.json"))
latest_raw = raw_files[-1]

print(f"Most recent raw file: {os.path.basename(latest_raw)}")
print(f"Total raw snapshots accumulated: {len(raw_files)}\n")

with open(latest_raw, "r") as f:
    raw_data = json.load(f)

print(f"Number of coins in latest snapshot: {len(raw_data)}")
print("\nSample record (first coin):")
print(json.dumps(raw_data[0], indent=2))

Most recent raw file: crypto_market_data_raw_2026-05-29.json
Total raw snapshots accumulated: 5

Number of coins in latest snapshot: 100

Sample record (first coin):
{
  "id": "bitcoin",
  "symbol": "btc",
  "name": "Bitcoin",
  "image": "https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400",
  "current_price": 73306,
  "market_cap": 1469300843946,
  "market_cap_rank": 1,
  "fully_diluted_valuation": 1469300843946,
  "total_volume": 32394691810,
  "high_24h": 73792,
  "low_24h": 72669,
  "price_change_24h": 4.13,
  "price_change_percentage_24h": 0.00564,
  "market_cap_change_24h": 405512604,
  "market_cap_change_percentage_24h": 0.02761,
  "circulating_supply": 20035990.0,
  "total_supply": 20035990.0,
  "max_supply": 21000000.0,
  "ath": 126080,
  "ath_change_percentage": -41.85724,
  "ath_date": "2025-10-06T18:57:42.558Z",
  "atl": 67.81,
  "atl_change_percentage": 108007.08285,
  "atl_date": "2013-07-06T00:00:00.000Z",
  "roi": null,
  "last_updated": "2026-

---
## Stage 2 — Processed Parquet Data

In [3]:
processed_df = spark.read.parquet(f"{BASE_PATH}/data/processed")

print(f"Total records in processed layer: {processed_df.count()}")
print(f"Number of partitions (dates): {processed_df.select('updated_date').distinct().count()}")
print("\nSchema:")
processed_df.printSchema()

Total records in processed layer: 300


Number of partitions (dates): 7

Schema:
root
 |-- ath: double (nullable = true)
 |-- ath_change_percentage: double (nullable = true)
 |-- ath_date: timestamp (nullable = true)
 |-- atl: double (nullable = true)
 |-- atl_change_percentage: double (nullable = true)
 |-- atl_date: timestamp (nullable = true)
 |-- circulating_supply: double (nullable = true)
 |-- current_price: double (nullable = true)
 |-- fully_diluted_valuation: long (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- id: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- market_cap: long (nullable = true)
 |-- market_cap_change_24h: double (nullable = true)
 |-- market_cap_change_percentage_24h: double (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- max_supply: double (nullable = true)
 |-- name: string (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = tru

In [4]:
print("Dates available in processed layer:")
processed_df.select("updated_date").distinct().orderBy("updated_date").show(truncate=False)

print("Sample processed records:")
processed_df.select(
    "name", "current_price", "market_cap", "total_volume",
    "circulating_supply", "updated_date"
).orderBy("market_cap", ascending=False).show(10, truncate=False)

Dates available in processed layer:
+------------+
|updated_date|
+------------+
|2026-05-18  |
|2026-05-21  |
|2026-05-23  |
|2026-05-24  |
|2026-05-25  |
|2026-05-26  |
|2026-05-28  |
+------------+

Sample processed records:


+--------+-------------+-------------+---------------+--------------------+------------+
|name    |current_price|market_cap   |total_volume   |circulating_supply  |updated_date|
+--------+-------------+-------------+---------------+--------------------+------------+
|Bitcoin |77910.0      |1560349334869|2.9194154601E10|2.00322E7           |2026-05-21  |
|Bitcoin |77265.0      |1547282708175|2.4931599587E10|2.0034078E7         |2026-05-25  |
|Bitcoin |73006.0      |1462654110065|4.2283794309E10|2.0035459E7         |2026-05-28  |
|Ethereum|2140.3       |258293665100 |1.1985654099E10|1.206856184990976E8 |2026-05-21  |
|Ethereum|2110.73      |254547653212 |1.1424936404E10|1.206855183533083E8 |2026-05-25  |
|Ethereum|1978.15      |238732576796 |1.6453920077E10|1.206854152818607E8 |2026-05-28  |
|Tether  |0.998993     |189650287168 |5.2537542889E10|1.898423595086838E11|2026-05-21  |
|Tether  |0.998814     |189412767655 |4.1870541741E10|1.896331457963721E11|2026-05-25  |
|Tether  |0.998269   

---
## Stage 4 — Analytics Outputs
The analytics job produces five output tables. We load and display each one below.

In [7]:
print("Average Market Cap Rankings:")
spark.read.parquet(f"{BASE_PATH}/data/analytics/avg_market_cap") \
    .orderBy("avg_market_cap_rank") \
    .show(10, truncate=False)

Average Market Cap Rankings:


+------------+---------------------+-------------------+
|name        |avg_market_cap       |avg_market_cap_rank|
+------------+---------------------+-------------------+
|Bitcoin     |1.523428717703E12    |1                  |
|Ethereum    |2.5052463170266666E11|2                  |
|Tether      |1.8943853906766666E11|3                  |
|BNB         |8.756601741133333E10 |4                  |
|XRP         |8.2789249592E10      |5                  |
|USDC        |7.634275805266667E10 |6                  |
|Solana      |4.8763510649666664E10|7                  |
|TRON        |3.4477458552666664E10|8                  |
|Figure Heloc|1.8649020024E10      |9                  |
|Dogecoin    |1.5720621958666666E10|10                 |
+------------+---------------------+-------------------+
only showing top 10 rows


In [8]:
print("Average Price Rankings:")
spark.read.parquet(f"{BASE_PATH}/data/analytics/avg_price") \
    .orderBy("avg_price_rank") \
    .show(10, truncate=False)

Average Price Rankings:
+------------+-------------+--------------+
|name        |average_price|avg_price_rank|
+------------+-------------+--------------+
|Bitcoin     |76060.333    |1             |
|PAX Gold    |4483.437     |2             |
|Tether Gold |4477.397     |3             |
|Ethereum    |2076.393     |4             |
|BNB         |649.663      |5             |
|Zcash       |624.71       |6             |
|Monero      |393.13       |7             |
|Bitcoin Cash|353.577      |8             |
|Bittensor   |270.28       |9             |
|OUSG        |115.34       |10            |
+------------+-------------+--------------+
only showing top 10 rows


In [9]:
print("Volume to Market Cap Ratio Rankings:")
spark.read.parquet(f"{BASE_PATH}/data/analytics/vol_market_ratio") \
    .orderBy("vol_market_rank") \
    .show(10, truncate=False)

Volume to Market Cap Ratio Rankings:
+--------------+----------------+---------------+---------------+
|name          |vol_market_ratio|total_volume   |vol_market_rank|
+--------------+----------------+---------------+---------------+
|USD1          |0.46663         |2.176174676E9  |1              |
|Dash          |0.39574         |2.50219036E8   |2              |
|Tether        |0.36955         |6.9938692752E10|3              |
|Worldcoin     |0.30361         |3.35274701E8   |4              |
|Injective     |0.29538         |1.60165753E8   |5              |
|Filecoin      |0.28554         |2.19380975E8   |6              |
|NEAR Protocol |0.26091         |8.15967096E8   |7              |
|Aave          |0.21651         |2.65961021E8   |8              |
|USDC          |0.21625         |1.6512426527E10|9              |
|Pudgy Penguins|0.21246         |1.24695729E8   |10             |
+--------------+----------------+---------------+---------------+
only showing top 10 rows


In [10]:
print("Current Top Coins by Price (latest snapshot):")
spark.read.parquet(f"{BASE_PATH}/data/analytics/curr_top_price") \
    .orderBy("current_price_rank") \
    .show(10, truncate=False)

Current Top Coins by Price (latest snapshot):
+-----------+-------------+-------------+---------------+---------------+------------------+------------+
|name       |current_price|market_cap   |market_cap_rank|total_volume   |current_price_rank|updated_date|
+-----------+-------------+-------------+---------------+---------------+------------------+------------+
|Bitcoin    |77910.0      |1560349334869|1              |2.9194154601E10|1                 |2026-05-21  |
|Bitcoin    |77265.0      |1547282708175|1              |2.4931599587E10|1                 |2026-05-25  |
|Bitcoin    |73006.0      |1462654110065|1              |4.2283794309E10|1                 |2026-05-28  |
|Circle USYC|1.12         |2995497586   |35             |112456.0       |1                 |2026-05-26  |
|PAX Gold   |4551.33      |2140319659   |43             |1.18750844E8   |2                 |2026-05-25  |
|PAX Gold   |4372.11      |2042404735   |43             |2.79361286E8   |2                 |2026-05-28  |


In [11]:
print("Top Performing Assets — Composite Ranking:")
spark.read.parquet(f"{BASE_PATH}/data/analytics/top_performing_assets") \
    .select(
        "top_performing_rank", "name", "avg_market_cap",
        "avg_market_cap_rank", "average_price", "avg_price_rank",
        "total_volume", "vol_market_ratio", "vol_market_rank",
        "top_performing_score"
    ) \
    .orderBy("top_performing_rank") \
    .show(10, truncate=False)

Top Performing Assets — Composite Ranking:
+-------------------+------------+---------------------+-------------------+-------------+--------------+---------------+----------------+---------------+--------------------+
|top_performing_rank|name        |avg_market_cap       |avg_market_cap_rank|average_price|avg_price_rank|total_volume   |vol_market_ratio|vol_market_rank|top_performing_score|
+-------------------+------------+---------------------+-------------------+-------------+--------------+---------------+----------------+---------------+--------------------+
|1                  |Ethereum    |2.5052463170266666E11|2                  |2076.393     |4             |1.6453920077E10|0.06892         |34             |9.0                 |
|2                  |Bitcoin     |1.523428717703E12    |1                  |76060.333    |1             |4.2283794309E10|0.02891         |61             |13.0                |
|2                  |Zcash       |1.0407415190333334E10|13                 |6

In [12]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
